# Scraping Quality Benchmark

This notebook presents a hybrid scraping system built to extract high quality content across a diverse set of URLs.

## Goals
- Maximize extraction of truth_text
- Minimize inclusion of lie_text
- Balance success rate, F1, and latency
- Support static pages, PDFs, and JavaScript heavy pages


## Assignment Summary

The dataset contains URLs with:
- truth_text which is the important content that should appear in the output
- lie_text which is navigation, boilerplate, and irrelevant text that should ideally be excluded

The system was evaluated on the training set using the provided `score.py` script. Test results were generated for final submission and are not evaluated locally because no ground truth is provided for the test set.


## Scraping Approaches

### Approach 1: Lightweight HTTP Fetch
- Uses `requests`
- Fast and inexpensive
- Handles HTML and PDFs
- Applies Trafilatura and heuristic filtering for cleanup

### Approach 2: Browser Rendering with Playwright
- Uses headless Chromium
- Handles JavaScript heavy pages
- Used as a fallback when the lightweight path fails or returns poor quality content

The final system combines both approaches in a fallback pipeline.


## Code Flow

```mermaid
flowchart TD
    A[Read URL from dataset] --> B[HTTP fetch without proxy]
    B --> C{Content good enough?}
    C -- Yes --> H[Finalize content]
    C -- No --> D[HTTP fetch with proxy]
    D --> E{Content good enough?}
    E -- Yes --> H
    E -- No --> F[Playwright without proxy]
    F --> G{Content good enough?}
    G -- Yes --> H
    G -- No --> I[Playwright with proxy]
    I --> H
    H --> J[Write JSONL result]
```


## Iterative Improvement

### Version 1
Basic HTTP fetching with raw extraction.  
Result: low precision because too much lie_text remained.

### Version 2
Added Trafilatura and heuristic cleanup.  
Result: better precision and content quality.

### Version 3
Added proxy support and PDF handling.  
Result: better coverage and improved success rate.

### Version 4
Added Playwright fallback for JavaScript heavy pages.  
Result: stronger recall and more robust scraping across page types.


In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt

TRAIN_RESULTS_PATH = "train_results.jsonl"

rows = []
with open(TRAIN_RESULTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["content_length"] = df["content"].fillna("").str.len()
df["successful"] = df["status_code"].between(200, 399) & (df["content_length"] > 0)

df.head()


,id,url,content,status_code,latency,content_length,successful
0,1,https://duckduckgo.com/?t=h_&q=DAIWIK+INVESTME...,,200,0.541,0,False
1,2,https://iqvia.com/solutions/technologies/orche...,IDC sees IQVIA One Home for Sites as well posi...,200,3.375,676,True
2,3,https://interpret.csis.org/translations/the-bu...,The outbreak of the Russia-Ukraine conflict in...,200,6.247,64944,True
3,4,https://fgsglobal.com/join-us/opportunities/pr...,,404,0.572,0,False
4,5,https://support.grip.events/e3t/Ctc/W+50013/d2...,How to add Grip to your Apple Developer Accoun...,200,4.299,1909,True


## Training Scores from `score.py`

The following values are the final training set scores produced by the provided scoring script.


In [2]:
score_summary = {
    "ground_truth_entries": 500,
    "result_entries": 500,
    "urls_evaluated": 500,
    "successful_scrapes": 323,
    "success_rate": 0.646,
    "overall_f1": 0.4066,
    "overall_precision": 0.4450,
    "overall_recall": 0.3917,
    "quality_f1": 0.6222,
    "quality_precision": 0.6778,
    "quality_recall": 0.5999,
    "latency_p50": 1.83,
    "latency_p90": 6.45,
    "latency_p95": 9.07,
    "latency_avg": 3.83,
}
pd.DataFrame([score_summary]).T.rename(columns={0: "value"})


,value
ground_truth_entries,500.0000
result_entries,500.0000
urls_evaluated,500.0000
successful_scrapes,323.0000
success_rate,0.6460
overall_f1,0.4066
overall_precision,0.4450
overall_recall,0.3917
quality_f1,0.6222
quality_precision,0.6778


## Interpretation

The final system achieved a success rate of 64.6 percent on the training set.

The most important quality result is the F1 score on successful scrapes, which reached 0.6222. This shows that when the scraper successfully retrieves a page, the extracted content usually aligns well with truth_text and keeps lie_text relatively low.

The main remaining gap is coverage rather than extraction quality. In other words, the system performs well on pages it can retrieve successfully, but some requests still fail due to blocking, dynamic content, or environment related issues.


## Final Method

The final method uses:
1. Lightweight HTTP fetch without proxy
2. Lightweight HTTP fetch with proxy
3. Playwright fallback without proxy
4. Playwright fallback with proxy

This provided the best balance between speed, robustness, and content quality.


## GitHub Repository

https://github.com/einavchazen/scraping-quality-benchmark
